# MICrONS: BOSPERRUS buffer vs. the Reimann heuristic

Left: degree vs. distance to the alpha-shape boundary of the neurons' x-z footprint, with the piecewise-linear
fit. Middle: mean degree per x-z bin (Reimann's 50 x 50 grid). Right: the same bins classified by the two analysis
buffers (BOSPERRUS: most of the bin's neurons closer than the total-degree knot; Reimann: the bin's max outer
synapse fraction > 5 %); bins kept by both coloured by mean degree. Data: `results/physically_realized_graphs/microns/`
(`src/physically_realized_graphs/microns_buffer.py`); plotting code: `microns_helpers.py`.

In [ ]:
import json

import bosperrus
import matplotlib.pyplot as plt
import microns_helpers as h
from matplotlib.patches import Patch

plt.rcParams["svg.fonttype"] = "none"
BSPBLUE = bosperrus.PiecewiseLinearFit.color
PALETTE = json.load(open("../../fit_palette.json"))
MANUAL_GREEN = PALETTE["Manual buffer"]  # as in meristem_L1_maps.svg
BOSPERRUS_ONLY = PALETTE["BOSPERRUS-only buffer"]
res = h.load()
s = res["summary"]
print(
    f"neurons affected: BOSPERRUS {s['n_bosperrus']:,}, Reimann per neuron {s['n_reimann_neuron']:,}, "
    f"per bin {s['n_reimann_bin']:,} (of {s['n_total']:,}); Jaccard vs. BOSPERRUS: per neuron "
    f"{s['jaccard_neuron']:.3f}, per bin {s['jaccard_bin']:.3f}"
)

In [ ]:
fig = plt.figure(figsize=(9, 2.9))  # at most 9 in wide; saved without bbox cropping so the size holds
outer = fig.add_gridspec(1, 2, width_ratios=[2, 5.4], wspace=0.22, left=0.07, right=0.93, top=0.88, bottom=0.3)
gs = outer[1].subgridspec(1, 3, width_ratios=[1, 1, 0.04], wspace=0.08)
ax_fit = fig.add_subplot(outer[0])
h.plot_fits_panel(
    ax_fit,
    res,
    measure="degree",
    fit_color=BSPBLUE,
    legend_label_fn=lambda b: f"piecewise-linear (knot {b / 1e3:.0f} µm)",
    ylabel="Degree",
    title="MICrONS mm³ connectome",
    log_y=False,
)
ax_deg, ax_cls = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
colors = {"both": BSPBLUE, "reimann_only": MANUAL_GREEN, "bosperrus_only": BOSPERRUS_ONLY, "no_value": "0.8"}
sm, counts = h.plot_bin_maps(ax_deg, ax_cls, res, "YlOrRd", colors, aspect=1.6)
ax_deg.set_title("mean degree per bin", fontsize=9)
ax_cls.set_title("analysis buffers", fontsize=9)
ax_fit.legend(fontsize=6, frameon=False, loc="upper left")
cb = fig.colorbar(sm, cax=fig.add_subplot(gs[2]))
cb.set_label("mean degree")
fig.legend(
    handles=[
        Patch(color=BSPBLUE, label=f"excluded by BOSPERRUS and Reimann ({counts['both']} bins)"),
        Patch(color=MANUAL_GREEN, label=f"excluded by Reimann only ({counts['reimann_only']})"),
        Patch(color=BOSPERRUS_ONLY, label=f"excluded by BOSPERRUS only ({counts['bosperrus_only']})"),
        Patch(color="0.8", label="no incoming synapses (no Reimann value)"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.62, 0.0),
    ncols=2,
    fontsize=7,
    frameon=False,
)
fig.savefig("microns.svg", dpi=300)